In [17]:
%pip install requests beautifulsoup4 pdfkit

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
import re
import requests
import pdfkit
from bs4 import BeautifulSoup
from pathlib import Path

In [ ]:
def get_books_and_links(toc_url):
    response = requests.get(toc_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    books = {}
    current_book = None

    tags = soup.find_all(["h2", "ul"])
    i = 0
    while i < len(tags):
        tag = tags[i]

        if tag.name == "h2" and "Book" in tag.get_text():
            current_book = tag.get_text(strip=True)
            books[current_book] = []

            if i + 1 < len(tags) and tags[i + 1].name == "ul":
                ul = tags[i + 1]

                chapter_lis = ul.find_all("li", recursive=False)
                if len(chapter_lis) == 1 and chapter_lis[0].find("ul"):
                    chapter_lis = chapter_lis[0].find("ul").find_all("li", recursive=False)

                for li in chapter_lis:
                    a = li.find("a", href=True)
                    if a:
                        title = a.get_text(strip=True)
                        href = a["href"]
                        books[current_book].append((title, href))

                i += 1  # Skip the UL

        i += 1

    return books


📘 Book 1 (30 chapters)
  - Prologue
  - Chapter 1: Knife
  - Chapter 2: Invitation
  - Chapter 3: Party
  - Chapter 4: Name
  - Chapter 5: Role
  - Chapter 6: Aspect
  - Chapter 7: Sword
  - Chapter 8: Introduction
  - Chapter 9: Claimant
  - Chapter 10: Menace
  - Chapter 11: Sucker Punch
  - Chapter 12: Squire
  - Chapter 13: Order
  - Chapter 14: Villain
  - Chapter 15: Company
  - Chapter 16: Game
  - Chapter 17: Set
  - Chapter 18: Match
  - Chapter 19: Pivot
  - Chapter 20: Rise
  - Chapter 21: Fall
  - Chapter 22: All According To
  - Chapter 23: Morok’s Plan
  - Chapter 24: Aisha’s Plan
  - Chapter 25: Snatcher’s Plan
  - Chapter 26: Juniper’s Plan
  - Chapter 27: Callow’s Plan
  - Chapter 28: Win Condition
  - Epilogue

📘 Book 2 (62 chapters)
  - Prologue
  - Chapter 1: Supply
  - Chapter 2: Demand
  - Chapter 3: Cost
  - Heroic Interlude: Balestra
  - Chapter 4: Return
  - Chapter 5: Recognition
  - Chapter 6: Rapport
  - Chapter 7: Reception
  - Chapter 8: Reversal
  - Chap

In [21]:
def sanitize_filename(title, index):
    title = re.sub(r'[^\w\s-]', '', title)
    title = re.sub(r'\s+', '_', title.strip())
    return f"Chapter_{index:02d}_{title}.pdf"

In [22]:
def save_books_as_pdfs(books, output_dir=".", wkhtmltopdf_path=None):
    if wkhtmltopdf_path:
        config = pdfkit.configuration(wkhtmltopdf=wkhtmltopdf_path)
    else:
        config = None

    output_dir = Path(output_dir)

    for book_title, chapters in books.items():
        print(f"\n📘 Saving {book_title}...")

        book_folder = output_dir / book_title.replace(" ", "_")
        book_folder.mkdir(exist_ok=True)

        for i, (chapter_title, url) in enumerate(chapters, start=1):
            file_name = sanitize_filename(chapter_title, i)
            file_path = book_folder / file_name
            print(f"  - Saving: {chapter_title} → {file_name}")

            try:
                pdfkit.from_url(url, str(file_path), configuration=config)
            except Exception as e:
                print(f"    ⚠️ Failed to save {chapter_title}: {e}")

In [ ]:
# Set this if wkhtmltopdf is not in PATH:
# wkhtmltopdf_path = r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe"
wkhtmltopdf_path = None  # Set to path if needed

toc_url = "https://practicalguidetoevil.wordpress.com/table-of-contents/"
#books = get_books_and_links(toc_url)

# Optional: test only on Book 1
books = {k: v for k, v in books.items() if k == "Book 1"}

save_books_as_pdfs(books, output_dir=".", wkhtmltopdf_path=wkhtmltopdf_path)

NotImplementedError: 